### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


# Figure 6.5: Precision and Recall vs Decision Threshold\n\nInteractive demo focused on the precision-recall trade-off as the decision threshold changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(42)


def make_scores(noise_scale=150_000, n_per_class=600):
    mu_pos, mu_neg = 180_000, -180_000
    pos = rng.normal(mu_pos, noise_scale, n_per_class)
    neg = rng.normal(mu_neg, noise_scale, n_per_class)
    scores = np.concatenate([pos, neg])
    y_true = np.concatenate([np.ones(n_per_class, dtype=int), np.zeros(n_per_class, dtype=int)])
    return scores, y_true

scores, y_true = make_scores()


def metrics_at_threshold(threshold):
    y_pred = (scores >= threshold).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall, tp, fp, fn


def plot_curves(threshold=0, noise_scale=150_000):
    global scores, y_true
    scores, y_true = make_scores(noise_scale=noise_scale)

    thresholds = np.linspace(-700_000, 700_000, 240)
    precision_vals = []
    recall_vals = []
    for t in thresholds:
        p, r, _, _, _ = metrics_at_threshold(t)
        precision_vals.append(p)
        recall_vals.append(r)

    p_now, r_now, tp, fp, fn = metrics_at_threshold(threshold)

    plt.figure(figsize=(10, 5.8))
    plt.plot(thresholds, precision_vals, 'b--', linewidth=2.6, label='Precision')
    plt.plot(thresholds, recall_vals, 'g-', linewidth=2.6, label='Recall')
    plt.axvline(threshold, color='k', linestyle=':', linewidth=1.8, label='Current threshold')
    plt.ylim(-0.02, 1.02)
    plt.xlabel('Threshold')
    plt.ylabel('Metric value')
    plt.title('Precision and Recall as Functions of the Decision Threshold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    print(f'Threshold = {threshold:,.0f}')
    print(f'Precision = {p_now:.3f}')
    print(f'Recall = {r_now:.3f}')
    print(f'TP = {tp}, FP = {fp}, FN = {fn}')


thr = widgets.IntSlider(
    description='threshold',
    min=-700_000,
    max=700_000,
    step=5_000,
    value=0,
    continuous_update=False,
)
noise = widgets.IntSlider(
    description='noise',
    min=60_000,
    max=300_000,
    step=5_000,
    value=150_000,
    continuous_update=False,
)

ui = widgets.HBox([thr, noise])
out = widgets.interactive_output(plot_curves, {'threshold': thr, 'noise_scale': noise})
display(ui, out)

